# Notebook 1: Data Inspection and Quality Checks

This notebook explores January 2024 NYC Yellow Taxi data and prepares clean, analytics-ready trip records.

## 1. Load raw trip data

Read the January 2024 Parquet file and preview sample records.

In [2]:
from pathlib import Path
import polars as pl

data_file = Path("../data/raw/yellow_tripdata_2024-01.parquet")

trips = pl.read_parquet(data_file)

trips.head(5)

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
i32,datetime[ns],datetime[ns],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,"""N""",186,79,2,17.7,1.0,0.5,0.0,0.0,1.0,22.7,2.5,0.0
1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.8,1,"""N""",140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0
1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.7,1,"""N""",236,79,1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0
1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.4,1,"""N""",79,211,1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0
1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.8,1,"""N""",211,148,1,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0


## 2. Inspect the schema

Review the source columns and data types before transforming the data.

In [4]:
trips.schema

Schema([('VendorID', Int32),
        ('tpep_pickup_datetime', Datetime(time_unit='ns', time_zone=None)),
        ('tpep_dropoff_datetime', Datetime(time_unit='ns', time_zone=None)),
        ('passenger_count', Int64),
        ('trip_distance', Float64),
        ('RatecodeID', Int64),
        ('store_and_fwd_flag', String),
        ('PULocationID', Int32),
        ('DOLocationID', Int32),
        ('payment_type', Int64),
        ('fare_amount', Float64),
        ('extra', Float64),
        ('mta_tax', Float64),
        ('tip_amount', Float64),
        ('tolls_amount', Float64),
        ('improvement_surcharge', Float64),
        ('total_amount', Float64),
        ('congestion_surcharge', Float64),
        ('Airport_fee', Float64)])

## 3. Calculate trip duration

Create `trip_duration_minutes` from pickup and drop-off timestamps.

In [ ]:
trips_with_duration = trips.with_columns(
    (
        pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime")
    )
    .dt.total_minutes()
    .alias("trip_duration_minutes")
)

trips_with_duration.select(
    [
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "trip_distance",
        "fare_amount",
        "total_amount",
        "trip_duration_minutes",
    ]
).head(5)

tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,total_amount,trip_duration_minutes
datetime[ns],datetime[ns],f64,f64,f64,i64
2024-01-01 00:57:55,2024-01-01 01:17:43,1.72,17.7,22.7,19
2024-01-01 00:03:00,2024-01-01 00:09:36,1.8,10.0,18.75,6
2024-01-01 00:17:06,2024-01-01 00:35:01,4.7,23.3,31.3,17
2024-01-01 00:36:38,2024-01-01 00:44:56,1.4,10.0,17.0,8
2024-01-01 00:46:51,2024-01-01 00:52:57,0.8,7.9,16.1,6


## 4. Load taxi-zone reference data

Load the lookup table that maps location IDs to borough and zone names.

In [7]:
zones = pl.read_csv("../data/reference/taxi_zone_lookup.csv")

zones = zones.with_columns(
            pl.col("LocationID").cast(pl.Int32)
        )
        
zones.head()

LocationID,Borough,Zone,service_zone
i32,str,str,str
1,"""EWR""","""Newark Airport""","""EWR"""
2,"""Queens""","""Jamaica Bay""","""Boro Zone"""
3,"""Bronx""","""Allerton/Pelham Gardens""","""Boro Zone"""
4,"""Manhattan""","""Alphabet City""","""Yellow Zone"""
5,"""Staten Island""","""Arden Heights""","""Boro Zone"""


## 5. Add pickup and drop-off locations

Join taxi-zone names to each trip.

In [8]:
pickup_zones = zones.select(
    [
        pl.col("LocationID").alias("PULocationID"),
        pl.col("Borough").alias("pickup_borough"),
        pl.col("Zone").alias("pickup_zone"),
    ]
)

dropoff_zones = zones.select(
    [
        pl.col("LocationID").alias("DOLocationID"),
        pl.col("Borough").alias("dropoff_borough"),
        pl.col("Zone").alias("dropoff_zone"),
    ]
)

trips_enriched = (
    trips_with_duration
    .join(pickup_zones, on="PULocationID", how="left")
    .join(dropoff_zones, on="DOLocationID", how="left")
)

trips_enriched.select(
    [
        "PULocationID",
        "pickup_borough",
        "pickup_zone",
        "DOLocationID",
        "dropoff_borough",
        "dropoff_zone",
        "trip_duration_minutes",
    ]
).head(10)

PULocationID,pickup_borough,pickup_zone,DOLocationID,dropoff_borough,dropoff_zone,trip_duration_minutes
i32,str,str,i32,str,str,i64
186,"""Manhattan""","""Penn Station/Madison Sq West""",79,"""Manhattan""","""East Village""",19
140,"""Manhattan""","""Lenox Hill East""",236,"""Manhattan""","""Upper East Side North""",6
236,"""Manhattan""","""Upper East Side North""",79,"""Manhattan""","""East Village""",17
79,"""Manhattan""","""East Village""",211,"""Manhattan""","""SoHo""",8
211,"""Manhattan""","""SoHo""",148,"""Manhattan""","""Lower East Side""",6
148,"""Manhattan""","""Lower East Side""",141,"""Manhattan""","""Lenox Hill West""",32
138,"""Queens""","""LaGuardia Airport""",181,"""Brooklyn""","""Park Slope""",26
246,"""Manhattan""","""West Chelsea/Hudson Yards""",231,"""Manhattan""","""TriBeCa/Civic Center""",28
161,"""Manhattan""","""Midtown Center""",261,"""Manhattan""","""World Trade Center""",28


## 6. Run data-quality checks

Identify invalid durations, negative amounts, negative distances, and missing zones.

In [9]:
quality_summary = trips_enriched.select(
    [
        pl.len().alias("total_trips"),
        (pl.col("trip_duration_minutes") <= 0)
        .sum()
        .alias("non_positive_duration"),

        (pl.col("trip_distance") < 0)
        .sum()
        .alias("negative_distance"),

        (pl.col("total_amount") < 0)
        .sum()
        .alias("negative_total_amount"),

        pl.col("pickup_zone")
        .is_null()
        .sum()
        .alias("missing_pickup_zone"),

        pl.col("dropoff_zone")
        .is_null()
        .sum()
        .alias("missing_dropoff_zone"),
    ]
)

quality_summary

total_trips,non_positive_duration,negative_distance,negative_total_amount,missing_pickup_zone,missing_dropoff_zone
u32,u32,u32,u32,u32,u32
2964624,35121,0,35504,0,0


In [10]:
invalid_trips = trips_enriched.filter(
    (pl.col("trip_duration_minutes") <= 0)
    | (pl.col("trip_distance") < 0)
    | (pl.col("total_amount") < 0)
    | pl.col("pickup_zone").is_null()
    | pl.col("dropoff_zone").is_null()
)

invalid_trips.select(
    [
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "trip_distance",
        "total_amount",
        "pickup_zone",
        "dropoff_zone",
        "trip_duration_minutes",
    ]
).head(10)

tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,total_amount,pickup_zone,dropoff_zone,trip_duration_minutes
datetime[ns],datetime[ns],f64,f64,str,str,i64
2024-01-01 00:52:09,2024-01-01 00:52:28,0.0,8.0,"""Upper East Side South""","""Upper East Side South""",0
2024-01-01 00:14:29,2024-01-01 00:14:29,0.0,8.0,"""Upper East Side North""","""N/A""",0
2024-01-01 00:08:48,2024-01-01 00:09:33,0.05,8.08,"""West Chelsea/Hudson Yards""","""East Chelsea""",0
2024-01-01 00:18:24,2024-01-01 00:30:39,2.16,-18.5,"""West Village""","""Two Bridges/Seward Park""",12
2024-01-01 00:26:02,2024-01-01 00:26:21,2.3,15.5,"""Central Harlem North""","""Central Harlem North""",0
2024-01-01 00:46:16,2024-01-01 00:46:31,0.02,74.0,"""Upper West Side South""","""Upper West Side South""",0
2024-01-01 00:04:00,2024-01-01 00:04:44,0.01,-34.25,"""Cypress Hills""","""Cypress Hills""",0
2024-01-01 00:04:00,2024-01-01 00:04:44,0.01,34.25,"""Cypress Hills""","""Cypress Hills""",0
2024-01-01 00:41:42,2024-01-01 00:46:00,0.47,-10.8,"""West Village""","""Greenwich Village North""",4


## 7. Classify valid and quarantined trips

Assign a rejection reason to invalid records and separate them from valid trips.

In [11]:
classified_trips = trips_enriched.with_columns(
    pl.when(pl.col("trip_duration_minutes") <= 0)
    .then(pl.lit("non_positive_duration"))

    .when(pl.col("trip_distance") < 0)
    .then(pl.lit("negative_distance"))

    .when(pl.col("total_amount") < 0)
    .then(pl.lit("negative_total_amount"))

    .when(pl.col("pickup_zone").is_null())
    .then(pl.lit("missing_pickup_zone"))

    .when(pl.col("dropoff_zone").is_null())
    .then(pl.lit("missing_dropoff_zone"))

    .otherwise(pl.lit(None))
    .alias("rejection_reason")
)

valid_trips = classified_trips.filter(
    pl.col("rejection_reason").is_null()
)

quarantine_trips = classified_trips.filter(
    pl.col("rejection_reason").is_not_null()
)

quarantine_trips.group_by("rejection_reason").len()

rejection_reason,len
str,u32
"""non_positive_duration""",35121
"""negative_total_amount""",30338


In [12]:
pl.DataFrame(
    {
        "dataset": ["valid_trips", "quarantine_trips"],
        "row_count": [valid_trips.height, quarantine_trips.height],
    }
)

dataset,row_count
str,i64
"""valid_trips""",2899165
"""quarantine_trips""",65459


## 8. Write output datasets

Save valid trips to the curated layer and invalid trips to the quarantine layer.

In [13]:
from pathlib import Path

curated_path = Path("../data/curated/yellow_tripdata_2024-01_curated.parquet")
quarantine_path = Path("../data/quarantine/yellow_tripdata_2024-01_quarantine.parquet")

curated_path.parent.mkdir(parents=True, exist_ok=True)
quarantine_path.parent.mkdir(parents=True, exist_ok=True)

valid_trips.write_parquet(curated_path, compression="zstd")
quarantine_trips.write_parquet(quarantine_path, compression="zstd")

print(f"Curated data written to: {curated_path}")
print(f"Quarantine data written to: {quarantine_path}")

Curated data written to: ..\data\curated\yellow_tripdata_2024-01_curated.parquet
Quarantine data written to: ..\data\quarantine\yellow_tripdata_2024-01_quarantine.parquet


## 9. Verify pipeline output

In [14]:
print(
    pl.scan_parquet(curated_path)
    .select(pl.len().alias("curated_rows"))
    .collect()
)

print(
    pl.scan_parquet(quarantine_path)
    .select(pl.len().alias("quarantine_rows"))
    .collect()
)

shape: (1, 1)
┌──────────────┐
│ curated_rows │
│ ---          │
│ u32          │
╞══════════════╡
│ 2899165      │
└──────────────┘
shape: (1, 1)
┌─────────────────┐
│ quarantine_rows │
│ ---             │
│ u32             │
╞═════════════════╡
│ 65459           │
└─────────────────┘
